# Protein Design Week Baseline Notebook

This notebook is a reusable baseline for hackathon day:
- validate your local environment,
- prepare and sanity-check protein sequences,
- connect to Hugging Face Spaces with `gradio_client`,
- keep outputs organized for PyMOL inspection.


## Orientation Checklist (Quick)

- PyMOL installed.
- Hugging Face account + org/form access complete.
- This kernel is from your `pdw` environment.
- You know which Space/API endpoint your team will use.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import requests
from gradio_client import Client

print("Python:", sys.version.split()[0])
print("Timestamp:", datetime.now().isoformat(timespec="seconds"))


/opt/homebrew/Caskroom/miniforge/base/envs/pdw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15
Timestamp: 2026-03-07T00:59:48


In [ ]:
# Keep local outputs deterministic and easy to inspect.
BASE = Path("output/pdw")
BASE.mkdir(parents=True, exist_ok=True)
(BASE / "fasta").mkdir(exist_ok=True)
(BASE / "results").mkdir(exist_ok=True)
(BASE / "logs").mkdir(exist_ok=True)
BASE


## Sequence Prep Helpers

Use this to validate amino-acid strings before submitting to design/prediction Spaces.


In [ ]:
AA20 = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq: str) -> str:
    seq = seq.strip().upper().replace(" ", "").replace("\n", "")
    return seq

def sequence_report(seq: str) -> dict:
    seq = clean_sequence(seq)
    invalid = sorted(set(ch for ch in seq if ch not in AA20))
    return {
        "length": len(seq),
        "invalid_chars": invalid,
        "is_valid_aa20": len(invalid) == 0,
    }

def write_fasta(seq: str, name: str, out_dir: Path) -> Path:
    seq = clean_sequence(seq)
    path = out_dir / f"{name}.fasta"
    path.write_text(f">{name}\n{seq}\n")
    return path


In [ ]:
# Replace with your team's sequence(s).
seed_seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANL"
rep = sequence_report(seed_seq)
rep


In [ ]:
fasta_path = write_fasta(seed_seq, name="seed_candidate", out_dir=BASE / "fasta")
fasta_path


## Hugging Face Space: API Discovery

Start here first. Confirm API schema before calling `predict`.


In [ ]:
# Example format: "owner/space-name"
SPACE_ID = "REPLACE_WITH_OWNER/REPLACE_WITH_SPACE"

# Optional token for gated/private Spaces:
# export HF_TOKEN=...
HF_TOKEN = os.getenv("HF_TOKEN")

if "REPLACE_WITH" in SPACE_ID:
    print("Set SPACE_ID before running this section.")
else:
    client = Client(SPACE_ID, hf_token=HF_TOKEN)
    api_info = client.view_api(all_endpoints=True)
    api_info


## Optional: Single API Call Template

Edit `api_name` and payload to match what `view_api()` reported.


In [ ]:
def safe_space_call(client: Client, *, api_name: str, **kwargs):
    try:
        result = client.predict(api_name=api_name, **kwargs)
        return {"ok": True, "result": result}
    except Exception as exc:
        return {"ok": False, "error": str(exc)}

# Example only; update with real endpoint + args.
# response = safe_space_call(
#     client,
#     api_name="/predict",
#     sequence=seed_seq,
# )
# response


## Minimal Result Tracking

Use a table so candidates and run metadata remain auditable during hackathon iterations.


In [ ]:
runs = [
    {
        "candidate_id": "seed_candidate",
        "sequence": seed_seq,
        "length": len(seed_seq),
        "space_id": SPACE_ID,
        "status": "draft",
        "notes": "baseline",
    }
]

df = pd.DataFrame(runs)
df


In [ ]:
run_table_path = BASE / "results" / "run_table.csv"
df.to_csv(run_table_path, index=False)
run_table_path


## PyMOL Validation Reminder

After structure output, check in PyMOL:
- mutation neighborhood clashes,
- interface contact plausibility,
- exposed/buried residue consistency with your design intent.

From orientation tutorial: use sequence view, selections, surface/stick views, and local contact inspection as first-pass QA.


## Tomorrow Start Protocol (5 min)

1. Run notebook top-to-bottom once.
2. Set `SPACE_ID` and verify `view_api()` works.
3. Submit one known-safe sequence.
4. Save outputs and record run metadata immediately.
5. Push promising candidates to PyMOL validation.
